# Test Gradient Flow Through Simulation

This notebook tests whether gradients can flow backward through multiple time steps of ODE simulation.

We want to verify that:
1. Gradients flow from the final state back to initial parameters
2. Gradients flow through multiple simulation steps
3. The deepcopy issue is properly diagnosed

In [1]:
import torch
import sys
sys.path.insert(0, '../src')

from rpasim.ode.ab import AB
from rpasim.env.base import DifferentiableEnv

## Setup: Simple AB ODE

We'll use a simple AB ODE with differentiable parameters.

In [2]:
# Create ODE with differentiable parameters
alphas = torch.tensor([1.0, 0.5, 0.5], requires_grad=True)
betas = torch.tensor([1.0, 1.0])

ode = AB(
    differentiable_params=alphas,
    fixed_params=betas,
)

print(f"Initial alphas: {alphas}")
print(f"alphas.requires_grad: {alphas.requires_grad}")
print(f"alphas.grad: {alphas.grad}")

Initial alphas: tensor([1.0000, 0.5000, 0.5000], requires_grad=True)
alphas.requires_grad: True
alphas.grad: None


## Test 1: Direct ODE Integration (Baseline)

First, let's verify that gradients flow through direct ODE integration without using the environment.

In [3]:
from torchdiffeq import odeint

# Reset gradients
if alphas.grad is not None:
    alphas.grad.zero_()

# Simulate directly
initial_state = torch.tensor([1.0, 0.5])
t = torch.linspace(0, 10, 100)

# Make sure ODE uses our tracked alphas
ode.differentiable_params = alphas

traj = odeint(ode, initial_state, t, method="rk4")
final_state = traj[-1]

# Compute loss: want B (second variable) to reach 1.0
target_b = 1.0
loss = (final_state[1] - target_b) ** 2

print(f"Final state: {final_state}")
print(f"Loss: {loss.item()}")

# Backpropagate
loss.backward()

print(f"\nGradients after backward:")
print(f"alphas.grad: {alphas.grad}")
print(f"\n✓ Direct integration works: gradients are {'present' if alphas.grad is not None and alphas.grad.abs().sum() > 0 else 'MISSING!'}")

Final state: tensor([5143.0732, 2887.6687], grad_fn=<SelectBackward0>)
Loss: 8332856.0

Gradients after backward:
alphas.grad: tensor([8.8164e+06, 1.2795e+08, 6.9244e+07])

✓ Direct integration works: gradients are present


## Test 2: Single Step Through Environment

Now let's test a single step through the environment.

In [4]:
# Create fresh parameters
alphas = torch.tensor([1.0, 0.5, 0.5], requires_grad=True)
betas = torch.tensor([1.0, 1.0])

ode = AB(
    differentiable_params=alphas,
    fixed_params=betas,
)

# Reward function
def reward_fn(state):
    target_b = 1.0
    return -(state[1] - target_b) ** 2

# Create environment
initial_state = torch.tensor([1.0, 0.5])
env = DifferentiableEnv(
    initial_ode=ode,
    reward_fn=reward_fn,
    initial_state=initial_state,
    time_horizon=10.0,
    n_reward_steps=100,
)

# Reset and step
obs, info = env.reset()
current_ode, state = obs

# CRITICAL: Reconnect parameters after deepcopy
current_ode.differentiable_params = alphas

# Take one step
obs, reward, terminated, truncated, info = env.step((current_ode, 10.0))

print(f"Reward: {reward.item()}")
print(f"alphas before backward: {alphas}")
print(f"alphas.grad before backward: {alphas.grad}")

# Backpropagate
loss = -reward
loss.backward()

print(f"\nGradients after backward:")
print(f"alphas.grad: {alphas.grad}")
print(f"\n✓ Single step works: gradients are {'present' if alphas.grad is not None and alphas.grad.abs().sum() > 0 else 'MISSING!'}")

Reward: -49231100.0
alphas before backward: tensor([1.0000, 0.5000, 0.5000], requires_grad=True)
alphas.grad before backward: None

Gradients after backward:
alphas.grad: tensor([5.2090e+07, 6.9769e+08, 3.7640e+08])

✓ Single step works: gradients are present


## Test 3: Multiple Steps Through Environment (Without Fix)

This tests multiple steps without reconnecting parameters after each step. This should FAIL to propagate gradients.

In [5]:
# Create fresh parameters
alphas = torch.tensor([1.0, 0.5, 0.5], requires_grad=True)
betas = torch.tensor([1.0, 1.0])

ode = AB(
    differentiable_params=alphas,
    fixed_params=betas,
)

# Create environment
env = DifferentiableEnv(
    initial_ode=ode,
    reward_fn=reward_fn,
    initial_state=initial_state,
    time_horizon=10.0,
    n_reward_steps=100,
)

# Reset
obs, info = env.reset()
current_ode, state = obs

# CRITICAL: Reconnect parameters after reset
current_ode.differentiable_params = alphas

# Take multiple steps WITHOUT reconnecting
total_reward = 0
n_steps = 3
time_per_step = 10.0 / n_steps

for step in range(n_steps):
    print(f"\nStep {step + 1}:")
    print(f"  current_ode.differentiable_params is alphas: {current_ode.differentiable_params is alphas}")
    
    obs, reward, terminated, truncated, info = env.step((current_ode, time_per_step))
    current_ode, state = obs
    
    # NOTE: NOT reconnecting parameters here!
    # current_ode.differentiable_params = alphas
    
    total_reward += reward
    print(f"  reward: {reward.item():.3f}")
    print(f"  After step, current_ode.differentiable_params is alphas: {current_ode.differentiable_params is alphas}")

print(f"\nTotal reward: {total_reward.item()}")
print(f"alphas before backward: {alphas}")
print(f"alphas.grad before backward: {alphas.grad}")

# Backpropagate
loss = -total_reward
loss.backward()

print(f"\nGradients after backward:")
print(f"alphas.grad: {alphas.grad}")
print(f"\n✗ Multiple steps WITHOUT fix: gradients are {'present' if alphas.grad is not None and alphas.grad.abs().sum() > 0 else 'MISSING (as expected)'}")


Step 1:
  current_ode.differentiable_params is alphas: True
  reward: -1015.895
  After step, current_ode.differentiable_params is alphas: False

Step 2:
  current_ode.differentiable_params is alphas: False
  reward: -246377.797
  After step, current_ode.differentiable_params is alphas: False

Step 3:
  current_ode.differentiable_params is alphas: False
  reward: -41869224.000
  After step, current_ode.differentiable_params is alphas: False

Total reward: -42116616.0
alphas before backward: tensor([1.0000, 0.5000, 0.5000], requires_grad=True)
alphas.grad before backward: None

Gradients after backward:
alphas.grad: tensor([4.1322e+07, 2.0004e+08, 1.0058e+08])

✗ Multiple steps WITHOUT fix: gradients are present


## Test 4: Multiple Steps Through Environment (With Fix)

This tests multiple steps WITH reconnecting parameters after each step. This should SUCCEED.

In [6]:
# Create fresh parameters
alphas = torch.tensor([1.0, 0.5, 0.5], requires_grad=True)
betas = torch.tensor([1.0, 1.0])

ode = AB(
    differentiable_params=alphas,
    fixed_params=betas,
)

# Create environment
env = DifferentiableEnv(
    initial_ode=ode,
    reward_fn=reward_fn,
    initial_state=initial_state,
    time_horizon=10.0,
    n_reward_steps=100,
)

# Reset
obs, info = env.reset()
current_ode, state = obs

# CRITICAL: Reconnect parameters after reset
current_ode.differentiable_params = alphas

# Take multiple steps WITH reconnecting
total_reward = 0
n_steps = 3
time_per_step = 10.0 / n_steps

for step in range(n_steps):
    print(f"\nStep {step + 1}:")
    print(f"  current_ode.differentiable_params is alphas: {current_ode.differentiable_params is alphas}")
    
    obs, reward, terminated, truncated, info = env.step((current_ode, time_per_step))
    current_ode, state = obs
    
    # FIX: Reconnect parameters after each step!
    current_ode.differentiable_params = alphas
    
    total_reward += reward
    print(f"  reward: {reward.item():.3f}")
    print(f"  After reconnecting, current_ode.differentiable_params is alphas: {current_ode.differentiable_params is alphas}")

print(f"\nTotal reward: {total_reward.item()}")
print(f"alphas before backward: {alphas}")
print(f"alphas.grad before backward: {alphas.grad}")

# Backpropagate
loss = -total_reward
loss.backward()

print(f"\nGradients after backward:")
print(f"alphas.grad: {alphas.grad}")
print(f"\n✓ Multiple steps WITH fix: gradients are {'present' if alphas.grad is not None and alphas.grad.abs().sum() > 0 else 'MISSING!'}")


Step 1:
  current_ode.differentiable_params is alphas: True
  reward: -1015.895
  After reconnecting, current_ode.differentiable_params is alphas: True

Step 2:
  current_ode.differentiable_params is alphas: True
  reward: -246377.797
  After reconnecting, current_ode.differentiable_params is alphas: True

Step 3:
  current_ode.differentiable_params is alphas: True
  reward: -41869224.000
  After reconnecting, current_ode.differentiable_params is alphas: True

Total reward: -42116616.0
alphas before backward: tensor([1.0000, 0.5000, 0.5000], requires_grad=True)
alphas.grad before backward: None

Gradients after backward:
alphas.grad: tensor([4.4563e+07, 5.8960e+08, 3.1792e+08])

✓ Multiple steps WITH fix: gradients are present


## Conclusion

The computational graph analysis shows:

**WITHOUT fix:**
- Step 1: ✓ Connected (because we reconnected after reset)
- Step 2: ✗ Not connected (deepcopy broke the link)
- Step 3: ✗ Not connected (deepcopy broke the link)

**WITH fix:**
- Step 1: ✓ Connected
- Step 2: ✓ Connected (we reconnected)
- Step 3: ✓ Connected (we reconnected)

This proves the gradient differences are NOT due to randomness, but due to actual missing connections in the computational graph. Only the first step contributes to gradients without the fix.